In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
import plotly.graph_objects as go

import utils
import network

sequence_length = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
complete_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike.csv')
print(complete_df.shape)
complete_df.head()

(81920326, 17)


,LCLid,tstp,energy(kWh/hh)_smoothed,energy(kWh/hh),mean,median,std,min,max,gradient,kmedoid_clusters,is_medoid,spike_type,spike_magnitude,temperature,windSpeed,humidity
0,MAC000002,2013-01-01 00:00:00,0.226083,0.219,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,6,0,0,1.0,0.33250,0.3672,0.6494
1,MAC000002,2013-01-01 00:30:00,0.222500,0.241,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,6,0,0,1.0,0.33885,0.3689,0.6429
2,MAC000002,2013-01-01 01:00:00,0.220000,0.191,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,6,0,0,1.0,0.34520,0.3706,0.6364
3,MAC000002,2013-01-01 01:30:00,0.208250,0.235,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,6,0,0,1.0,0.34085,0.3784,0.6104
4,MAC000002,2013-01-01 02:00:00,0.198167,0.182,0.252475,0.212667,0.15878,0.0,1.65425,0.109262,6,0,0,1.0,0.33650,0.3862,0.5844


In [5]:
ids = complete_df['LCLid'].unique()

In [9]:

train_ids = np.random.choice(ids, 1, replace=False)

train_df = complete_df[complete_df['LCLid'].isin(train_ids)]

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_df['tstp'], y=train_df['energy(kWh/hh)'], mode='lines', name='Energy consumption'))

fig.update_layout(title='Energy consumption', xaxis_title='Time', yaxis_title='Energy consumption (kWh/hh)')
fig.show()

: 

In [23]:
train_df['date'] = pd.to_datetime(train_df['tstp']).dt.date
print(train_df.head())

              LCLid                 tstp  energy(kWh/hh)_smoothed  \
40198651  MAC002776  2013-01-01 00:00:00                 0.370333   
40198652  MAC002776  2013-01-01 00:30:00                 0.302833   
40198653  MAC002776  2013-01-01 01:00:00                 0.272083   
40198654  MAC002776  2013-01-01 01:30:00                 0.241750   
40198655  MAC002776  2013-01-01 02:00:00                 0.197917   

          energy(kWh/hh)      mean    median       std     min    max  \
40198651           0.378  0.296629  0.238083  0.213182  0.0345  1.543   
40198652           0.353  0.296629  0.238083  0.213182  0.0345  1.543   
40198653           0.355  0.296629  0.238083  0.213182  0.0345  1.543   
40198654           0.297  0.296629  0.238083  0.213182  0.0345  1.543   
40198655           0.125  0.296629  0.238083  0.213182  0.0345  1.543   

          gradient  kmedoid_clusters  is_medoid  spike_type  spike_magnitude  \
40198651  0.129487                 6          0           0       

C:\Users\edwar\AppData\Local\Temp\ipykernel_18484\3053014960.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [26]:
# Only keep the date and energy consumption columns
train_df = train_df[['date', 'energy(kWh/hh)']]
train_df = train_df.groupby('date').mean().reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(x=train_df['date'], y=train_df['energy(kWh/hh)'], mode='lines', name='Energy consumption trend'))

fig.update_layout(title='Energy consumption trend', xaxis_title='Date', yaxis_title='Average daily energy consumption (kWh/hh)')
fig.show()

In [22]:
def get_weather_df():
    """
    Returns the weather dataframe
    """
    weather_df = pd.read_csv("../../Data_raw/weather_hourly_darksky.csv")
    weather_df['time'] = pd.to_datetime(weather_df['time'])
    weather_df.sort_values(by='time', inplace=True)
    return weather_df

weather_df = get_weather_df()
weather_df.set_index('time', inplace=True)
weather_df = weather_df.reset_index()
print(weather_df.head())

                 time  visibility  windBearing  temperature  dewPoint  pressure  apparentTemperature  windSpeed precipType                 icon  humidity        summary
0 2011-11-01 00:00:00       13.63          160        13.49     11.48   1008.14                13.49       3.11       rain          clear-night      0.88          Clear
1 2011-11-01 01:00:00       13.26          154        12.73     11.58   1007.88                12.73       3.08       rain  partly-cloudy-night      0.93  Partly Cloudy
2 2011-11-01 02:00:00       12.94          161        13.65     12.14   1007.09                13.65       3.71       rain          clear-night      0.91          Clear
3 2011-11-01 03:00:00       12.99          170        14.13     12.24   1006.50                14.13       3.95       rain  partly-cloudy-night      0.88  Partly Cloudy
4 2011-11-01 04:00:00       12.92          180        14.17     12.59   1006.14                14.17       3.97       rain  partly-cloudy-night      0.90  

In [23]:
weather_df.set_index('time', inplace=True)
weather_df = weather_df.drop(columns=['precipType', 'icon', 'summary'])
weather_df = weather_df.resample('30T').mean().interpolate()
weather_df = weather_df.reset_index()
print(weather_df.head())

                 time  visibility  windBearing  temperature  dewPoint  pressure  apparentTemperature  windSpeed  humidity
0 2011-11-01 00:00:00      13.630        160.0        13.49     11.48  1008.140                13.49      3.110     0.880
1 2011-11-01 00:30:00      13.445        157.0        13.11     11.53  1008.010                13.11      3.095     0.905
2 2011-11-01 01:00:00      13.260        154.0        12.73     11.58  1007.880                12.73      3.080     0.930
3 2011-11-01 01:30:00      13.100        157.5        13.19     11.86  1007.485                13.19      3.395     0.920
4 2011-11-01 02:00:00      12.940        161.0        13.65     12.14  1007.090                13.65      3.710     0.910


: 